# Phase 6 — physical results: measured vs predicted

Post-processes the **bench measurements** of the printed top-k designs (IMU, anemometer, microphone) against the optimization's predictions. Runs **locally on the M3** (pure-numpy reduction — no SU2/gmsh).

Reads `recommended.json` (which designs were printed + their predicted `I_wrist` / 3D `J_fan`) and per-design measurement folders `<measurements>/<design>/{imu,anemometer,acoustic}.csv`. Thin orchestration — logic lives in tested `fanopt.physical.calibration` + `scripts/run_phase6_physical.py`. Re-run as measurements come in; unmeasured designs show as *pending* (nothing is invented).

## 1. Repo path + imports

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path.cwd()
while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / "pyproject.toml").exists():
    REPO_DIR = REPO_DIR.parent
for p in (REPO_DIR / "src", REPO_DIR / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import matplotlib.pyplot as plt

import run_phase6_physical as script
print("repo:", REPO_DIR)

## 2. Settings

Point these at the Phase-6 recommendation and where the operator saved the bench CSVs.

In [ ]:
RECOMMENDED = REPO_DIR / "data" / "phase6_recommend" / "recommended.json"
MEASUREMENTS = REPO_DIR / "data" / "phase6_measurements"
OUT_DIR = REPO_DIR / "data" / "phase6_recommend"

assert RECOMMENDED.exists(), (
    f"no recommended.json at {RECOMMENDED} — run scripts/recommend_designs.py first "
    "(it consumes the Phase-4 campaign + Phase-5 verification)."
)
print("recommended:", RECOMMENDED, "\nmeasurements:", MEASUREMENTS)

## 3. Reduce + calibrate

In [ ]:
report = script.run(recommended_path=RECOMMENDED, measurements_dir=MEASUREMENTS, out_dir=OUT_DIR)

print("designs:", report["n_designs"],
      "| measured  imu:", report["n_with_imu"],
      "anemometer:", report["n_with_anemometer"],
      "acoustic:", report["n_with_acoustic"])
print("J_fan rank vs CFD prediction:", report["j_fan_rank"])
report["designs"]

## 4. Measured observables per design

`W_cycle` (wrist work), `J_fan_proxy` (anemometer thrust), and SPL side by side. The **J_fan rank** panel is the calibration check — do the measured thrusts order the designs the same way the CFD `J_fan` did? (τ > 0 ⇒ yes.)

In [ ]:
ds = report["designs"]
names = [d["name"] for d in ds]


def _bar(ax, key, title, unit):
    vals = [d[key] for d in ds]
    xs = [i for i, v in enumerate(vals) if v is not None]
    ys = [vals[i] for i in xs]
    ax.bar(xs, ys, color="steelblue")
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8)
    ax.set_title(title)
    ax.set_ylabel(unit)
    for i, d in enumerate(ds):  # mark pending designs
        if d[key] is None:
            ax.text(i, 0, "pending", rotation=90, va="bottom", ha="center", fontsize=7, color="gray")


fig, axes = plt.subplots(1, 4, figsize=(15, 4))
_bar(axes[0], "w_cycle_j", "IMU W_cycle", "J / cycle")
_bar(axes[1], "j_fan_proxy_n", "Anemometer J_fan_proxy", "N")
_bar(axes[2], "spl_db", "Acoustic SPL", "dB")

# J_fan calibration: measured proxy vs predicted 3D J_fan
axp = axes[3]
paired = [(d["predicted_j_fan_3d"], d["j_fan_proxy_n"], d["name"]) for d in ds
          if d["predicted_j_fan_3d"] is not None and d["j_fan_proxy_n"] is not None]
if paired:
    axp.scatter([p[0] for p in paired], [p[1] for p in paired], s=70)
    for pj, mj, nm in paired:
        axp.annotate(nm, (pj, mj), fontsize=8, xytext=(4, 4), textcoords="offset points")
r = report["j_fan_rank"]
axp.set_title(f"J_fan calibration | tau={r.get('kendall_tau')} preserved={r.get('rank_preserved')}")
axp.set_xlabel("predicted 3D J_fan (nondim)")
axp.set_ylabel("measured J_fan_proxy (N)")
fig.tight_layout()
plt.show()